
Script to find TFs with available PWM in JASPAR DB for S. domuncula based on the orthology table

In [1]:
import pandas as pd
from pyjaspar import jaspardb

In [2]:
# specify JASPAR release
jdb_obj = jaspardb(release='JASPAR2024')

In [5]:
# New genome, SUb4
translation = pd.read_excel("...\\jaspar_meme_name_id.xlsx")
# hs
orthology_1 = pd.read_excel("...\\sd_v4_hs_orthology_250818.xlsx")
# mm
orthology_2 = pd.read_excel("...\\sd_v4_mm_orthology_250909.xlsx")


In [8]:
# for ind in transcript_df.index:
def get_TFs(transcript_df, df_translation, gene_n_col, output_name='_', output_id='_',save=False):
    motif_one_line = ''
    motifs_list = []
    motifs_list_upp = []
    motif_id_list = []
    transcript_df["matrix_id"] = pd.Series(dtype="object")  #
    transcript_df["tf_name"] = pd.Series(dtype="object")
    for ind in transcript_df.index:

        if transcript_df["blast_res"][ind] == "error":
            continue

        for tf_ind in df_translation.index:
            if str(transcript_df[gene_n_col][ind]).upper() == df_translation['name'][tf_ind].upper():
                transcript_df.loc[ind, "matrix_id"] = df_translation['matrix_id'][tf_ind]
                transcript_df.loc[ind, "tf_name"] = df_translation['name'][tf_ind]
                motifs_list.append(transcript_df[gene_n_col][ind])
                motifs_list_upp.append(transcript_df[gene_n_col][ind].upper())
                motif_id_list.append(transcript_df["matrix_id"][ind])
                break

    print('number of detected PWMs: ', len(motifs_list))

    if save:
        # sort both lists
        paired = sorted(zip(motifs_list_upp, motif_id_list))

        sorted_motifs_list_upp, sorted_motif_id_list = map(list, zip(*paired))

        with open(output_name, "w") as outfile:
            outfile.write("\n".join(sorted_motifs_list_upp))

        with open(output_id, "w") as outfile:
            outfile.write("\n".join(sorted_motif_id_list))

    return  motifs_list, motifs_list_upp, motif_id_list

def compare_sets(transcriptome_1, transcriptome_2, df_translation, gene_name_col_1, gene_name_col_2, out_1='_', out_2='_', save=False):
    tf_1, tf_1_upp, tf_1_id = get_TFs(transcriptome_1, df_translation, gene_name_col_1)
    tf_2, tf_2_upp, tf_2_id = get_TFs(transcriptome_2, df_translation, gene_name_col_2)

    tf_set_1_upp, tf_set_2_upp = set(tf_1_upp), set(tf_2_upp)
    print('number of unique detected PWMs:\nset 1: ', len(tf_set_1_upp), ', set 2 : ', len(tf_set_2_upp))

    union = tf_set_1_upp | tf_set_2_upp
    print('\ncombined number of unique TFs (union): ', len(union)) #, '\n', list(union)

    intersection = tf_set_1_upp & tf_set_2_upp
    print('\nnumber of unique TFs in both sets (intersection): ', len(intersection)) # , '\n', list(intersection)

    print('\nunique TFs found in 1 but not in 2: ', len(tf_set_1_upp - tf_set_2_upp), '\n', list(tf_set_1_upp - tf_set_2_upp))

    print('\nunique TFs found in 2 but not in 1: ', len(tf_set_2_upp - tf_set_1_upp), '\n', list(tf_set_2_upp - tf_set_1_upp))

    return tf_1_upp, tf_2_upp, union, intersection



In [7]:
TF_list_upp_1, TF_list_upp_2, TF_union, TF_intersect = compare_sets(transcriptome_1=orthology_1,
                                                                    transcriptome_2=orthology_1,
                                                                    df_translation=translation,
                                                                    gene_name_col_1="Gene Symbol",
                                                                    gene_name_col_2="Gene Symbol")

number of detected PWMs:  121
number of detected PWMs:  114
number of unique detected PWMs:
set 1:  93 , set 2 :  89

combined number of unique TFs :  78

unique TFs found in 1 but not in 2:  15 
 ['ZBED1', 'ARX', 'PBX1', 'WT1', 'NR5A2', 'KLF13', 'ZNF410', 'MEF2A', 'ARID3A', 'TBX19', 'CREB1', 'HIF1A', 'TFE3', 'TFDP1', 'TP73']

unique TFs found in 2 but not in 1:  11 
 ['MEF2C', 'SREBF1', 'ETV6', 'TBP', 'ZBED4', 'MITF', 'HSF2', 'FEV', 'CREM', 'NR2E3', 'ALX4']


In [ ]:
TF_list, TF_list_upp, TF_id_list = get_TFs(transcript_df=,
                               df_translation=,
                               gene_n_col=,
                               output_name='_',
                               output_id='_',
                               save=False)

In [7]:
# check which TFs are found more than once
df_relevant = translation[translation['name'].isin(TF_id_list)]
repeats = df_relevant[df_relevant['name'].duplicated(keep=False)]
print(repeats['name'].unique())
repeats

,Unnamed: 0,matrix_id,name,consensus
22,22,MA0596.1,SREBF2,ATGGGGTGAT
40,40,MA0669.1,NEUROG2,AACATATGTC
48,48,MA0691.1,TFAP4,AACAGCTGAT
94,94,MA0838.1,CEBPG,ATTGCGCAAT
130,130,MA1570.1,TFAP4,AACATATGTT
261,261,MA1636.2,CEBPG,ATGATGCAAT
323,323,MA0828.3,SREBF2,ATCACGTGAT
380,380,MA0492.2,JUND,GATGATGTCAT
381,381,MA0491.3,JUND,ATGACTCAT
462,462,MA1991.2,HNF1A,CCTTTGATCT
